In [ ]:
print("Hello")

In [ ]:
import pandas as pd
data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")

In [ ]:
data

In [ ]:
pd.unique(data['equip_type_famille'])

In [ ]:
data['equip_type_famille'].str.lower().str.contains("site d'activités aquatiques et nautiques").sum()

In [44]:

data_nautique = data[data['equip_type_famille'] == "Site d'activités aquatiques et nautiques"]


colonnes = ['equip_numero', "inst_numero", "inst_nom", "inst_adresse", "inst_cp", "dep_code", "reg_code", "lib_bdv", "equip_nom", "equip_type_name", "equip_coordonnees", "aps_name"]
data_nautique = data_nautique[colonnes].reset_index(drop=True).copy()



In [42]:
data_sud = data_nautique[data_nautique['dep_code'].str.lower().isin(["13", "6"])]
data_sud['aps_name'].str.lower().str.contains("surf|ski|voile|foil|wake").sum()

99

In [48]:
mots = ["surf", "ski", "voile", "foil", "wake"]
pattern = "|".join(mots)
data_nautique = data_nautique[data_nautique['aps_name'].str.contains(pattern, case=False, na=False)]

data_nautique

,equip_numero,inst_numero,inst_nom,inst_adresse,inst_cp,dep_code,reg_code,lib_bdv,equip_nom,equip_type_name,equip_coordonnees,aps_name,lat,lon
4,E003I974130041,I974130041,Baie de St Leu,rue de la Campagnie des Indes,97416,974,4.0,Saint-Leu,La Cafrine,Site d'activités aquatiques et nautiques,"-21.1867, 55.2866",Surf,-21.186700,55.286600
5,E003I974130046,I974130046,Cimetière de St Leu,RN 1 Grand Fond,97416,974,4.0,Saint-Leu,Le cimetière,Site d'activités aquatiques et nautiques,"-21.1858, 55.287",Surf,-21.185800,55.287000
8,E001I690640001,I690640001,BASE DE LOISIRS DE CONDRIEU,LA PRESQU'ILE,69420,69,84.0,Condrieu,Base de loisirs-plan d'eau,Site d'activités aquatiques et nautiques,"45.45552, 4.77446","Canoë de randonnée,Téléski nautique",45.455520,4.774460
15,E004I011840003,I011840003,CHAMBOD - CENTRE UDPA,NaN,1250,1,84.0,Ceyzériat,SITE MIXTE,Site d'activités aquatiques et nautiques,"46.12417, 5.42917","Baignade loisirs,Canoë de randonnée,Kayak-polo...",46.124170,5.429170
17,E005I412690005,I412690005,COMPLEXE SPORTIF DES GRANDS PRES,Rue Geoffroy Martel,41100,41,24.0,Vendôme,Base nautique des Grands Prés,Site d'activités aquatiques et nautiques,"47.79099, 1.0741","Canoë de randonnée,Planche à Voile",47.790990,1.074100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11570,E006I581800003,I581800003,Base Nautique des Settons,Lac des Settons D 193 Montsauche les settons,58230,58,27.0,Saulieu,Base de plein air multiactivités,Site d'activités aquatiques et nautiques,"47.18739, 4.0643","Aviron,Canoë de randonnée,Dériveur / Multicoqu...",47.187390,4.064300
11576,E006I721820001,I721820001,Espace touristique du lac de Mansigné,Route de Plessis,72510,72,52.0,Écommoy,Plan d'eau,Site de pêche,"47.75788, 0.13017",Dériveur / Multicoques / Courses océaniques / ...,47.757880,0.130170
11583,E004I880820003,I880820003,Pôle sports nature,La grande haie,88110,88,44.0,Raon-l'Étape,Site d'activités aquatiques,Site d'activités aquatiques et nautiques,"48.4565, 6.95106","Baignade loisirs,Canoë de randonnée,Planche à ...",48.456500,6.951060
11585,E004I890240043,I890240043,Base nautique de VAUX,22 Rue du poiry Vaux,89000,89,27.0,Auxerre,Base de ski nautique - Rte de Poiry,Stade de ski nautique,"47.76078, 3.59631",Ski nautique classique / Course / à figures li...,47.760780,3.596310


In [46]:
import geopandas as gpd
from shapely.geometry import Point

# Séparer lat/lon (si equip_coordonnees est du type "lat, lon")
data_nautique[['lat', 'lon']] = data_nautique['equip_coordonnees'].str.split(',', expand=True).astype(float)

data_nautique['lat'] = data_nautique['lat'].dropna()
data_nautique['lon'] = data_nautique['lon'].dropna()
data_nautique['equip_nom'] = data_nautique['equip_nom'].dropna()
# Créer le GeoDataFrame
geometry = [Point(lon, lat) for lat, lon in zip(data_nautique['lat'], data_nautique['lon'])]
gdf = gpd.GeoDataFrame(data_nautique, geometry=geometry, crs=2154)

In [47]:


import folium
m = folium.Map(location=[46.6, 2.3], zoom_start=6)

for _, row in data_nautique.dropna(subset=['lat', 'lon']).iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=row['aps_name']
    ).add_to(m)

m
